## Goal

> ⚠️ **Research on public allocation data.** Not investment advice; results describe the methods, not the funds.

> 🇸🇪 **Categories and cluster names are in Swedish**, as delivered by the source. Common words: *Branschfond* = sector fund, *Sverige* = Sweden, *Norden* = Nordics, *Små-/medelstora bolag* = small/mid caps, *Mix bolag* = mixed, *Råvaror* = commodities, *Läkemedel* = pharma.

What happened to each **cluster** from week to week? `cluster_migration.ipynb` follows funds between two weeks; this notebook follows the clusters themselves through every week in `WEEKS`, Hellinger only.

## How transitions are labelled

Cluster ids are renumbered on every run, so clusters are followed by the **funds they share** (only funds present in both weeks count). A *piece* is at least a quarter of a cluster's funds. This is the MONIC approach (Spiliopoulou et al., 2006).

| Week | Transition | Meaning |
|---|---|---|
| earlier | `survived` | More than half its funds went into one cluster, and they are more than half of it |
| earlier | `split` | Two or more pieces, none a survival |
| earlier | `absorbed` | Most of its funds went into one cluster, but they are a minority there |
| earlier | `disappeared` | Its funds scattered or left clusters |
| later | `continued` | One earlier cluster makes up a piece of it |
| later | `merged` | Two or more earlier clusters each make up a piece of it |
| later | `new` | No earlier cluster does: built from funds outside a cluster or new funds |

In [ ]:
# Re-import edited fund_clustering modules automatically before each cell runs.
%load_ext autoreload
%autoreload 2

In [ ]:
# Weeks to follow, in order. Each must be run first, one week at a time:
#   python main.py --week 2026-W28   (reads data/, writes output/)
from pathlib import Path

from fund_clustering.migration import load_method_week

OUTPUT_DIR = Path("../output")  # main.py output for every week below
METHOD_ID = "hellinger"
WEEKS = [
    "2026-W28", "2026-W29", "2026-W30", "2026-W31", "2026-W32", "2026-W33",
    "2026-W34", "2026-W35", "2026-W36", "2026-W37", "2026-W38", "2026-W39",
    "2026-W40", "2026-W41", "2026-W42", "2026-W43", "2026-W44", "2026-W45",
]

weeks = {week: load_method_week(OUTPUT_DIR, METHOD_ID, week) for week in WEEKS}
pairs = list(zip(WEEKS, WEEKS[1:]))

len(weeks), len(pairs)

In [ ]:
# Transitions for every consecutive pair, plus cluster names and sizes per week.
import pandas as pd

from fund_clustering.cluster import NOT_IN_CLUSTER
from fund_clustering.migration import cluster_transitions

names = {w: weeks[w].points.groupby("cluster")["cluster_name"].first() for w in WEEKS}
sizes = {w: weeks[w].points["cluster"].value_counts() for w in WEEKS}

transitions = {
    (a, b): cluster_transitions(weeks[a].points["cluster"], weeks[b].points["cluster"])
    for a, b in pairs
}


def week_row(a: str, b: str) -> dict:
    t = transitions[(a, b)]
    counts = t.groupby(["side", "transition"]).size()
    labels = weeks[b].points["cluster"]
    return {
        "week": b,
        "funds": len(labels),
        "clusters": labels[labels != NOT_IN_CLUSTER].nunique(),
        **{k: int(counts.get(("before", k), 0)) for k in
           ["survived", "split", "absorbed", "disappeared"]},
        **{k: int(counts.get(("after", k), 0)) for k in ["merged", "new"]},
    }


# One row per week: what happened since the week before.
overview = pd.DataFrame([week_row(a, b) for a, b in pairs]).set_index("week")
overview

## Each week

One Sankey per week: last week's clusters on the left, this week's on the right. Bands are coloured by what happened to the left cluster; grey means it survived. Bands under `MIN_FLOW` funds are left out, and funds outside a cluster are shown as "Not in a cluster".

In [ ]:
# Colours per transition of the earlier cluster.
import plotly.graph_objects as go

MIN_FLOW = 2

TRANSITION_COLOURS = {
    "survived": "rgba(154,160,166,0.45)",
    "split": "rgba(197,131,0,0.7)",
    "absorbed": "rgba(52,120,198,0.6)",
    "disappeared": "rgba(208,53,58,0.7)",
    "not in a cluster": "rgba(154,160,166,0.2)",
}


def label(week: str, cluster: int) -> str:
    if cluster == NOT_IN_CLUSTER:
        return "Not in a cluster"
    return f"{names[week][cluster]} ({sizes[week][cluster]})"


def week_flows(a: str, b: str) -> pd.DataFrame:
    """Funds per (earlier cluster, later cluster), with the earlier cluster's transition."""
    shared = weeks[a].points.index.intersection(weeks[b].points.index)
    pairs_ = pd.DataFrame(
        {"before": weeks[a].points.loc[shared, "cluster"],
         "after": weeks[b].points.loc[shared, "cluster"]}
    )
    flows = pairs_.value_counts().rename("funds").reset_index()
    before_side = transitions[(a, b)].query("side == 'before'")
    kind = dict(zip(before_side["cluster"], before_side["transition"]))
    flows["transition"] = flows["before"].map(kind).fillna("not in a cluster")
    return flows[flows["funds"] >= MIN_FLOW]


def week_sankey(a: str, b: str) -> go.Figure:
    flows = week_flows(a, b)
    left = sorted(flows["before"].unique())
    right = sorted(flows["after"].unique())
    index_left = {c: i for i, c in enumerate(left)}
    index_right = {c: len(left) + i for i, c in enumerate(right)}
    events = overview.loc[b, ["split", "absorbed", "disappeared", "merged", "new"]]
    events = ", ".join(f"{n} {k}" for k, n in events.items() if n) or "no changes"
    fig = go.Figure(
        go.Sankey(
            node=dict(
                label=[label(a, c) for c in left] + [label(b, c) for c in right],
                pad=6, thickness=10, color="#5f6368",
            ),
            link=dict(
                source=flows["before"].map(index_left),
                target=flows["after"].map(index_right),
                value=flows["funds"],
                color=flows["transition"].map(TRANSITION_COLOURS),
                customdata=flows["transition"],
                hovertemplate="%{value} funds, %{customdata}<extra></extra>",
            ),
        )
    )
    fig.update_layout(
        title=f"{a} → {b}: {events}",
        height=max(600, 16 * max(len(left), len(right))),
    )
    return fig


for a, b in pairs:
    week_sankey(a, b).show()

## All weeks at once

Every week as a column, clusters with at least `MIN_SIZE` funds only. Follow a coloured band to see where a cluster split, was absorbed or disappeared. Hover a node for its name.

In [ ]:
# Alluvial over all WEEKS: nodes are (week, cluster), links are consecutive-week flows.
MIN_SIZE = 10

nodes = [
    (w, c) for w in WEEKS for c in sorted(sizes[w].index)
    if c != NOT_IN_CLUSTER and sizes[w][c] >= MIN_SIZE
]
node_index = {node: i for i, node in enumerate(nodes)}

links = pd.concat(
    [week_flows(a, b).assign(a=a, b=b) for a, b in pairs], ignore_index=True
)
links = links[
    [(a, x) in node_index and (b, y) in node_index
     for a, b, x, y in zip(links["a"], links["b"], links["before"], links["after"])]
]

fig = go.Figure(
    go.Sankey(
        arrangement="snap",
        node=dict(
            label=["" for _ in nodes],
            customdata=[f"{w}: {label(w, c)}" for w, c in nodes],
            hovertemplate="%{customdata}<extra></extra>",
            pad=4, thickness=8, color="#5f6368",
            x=[WEEKS.index(w) / (len(WEEKS) - 1) * 0.98 + 0.01 for w, _ in nodes],
        ),
        link=dict(
            source=[node_index[(a, x)] for a, x in zip(links["a"], links["before"])],
            target=[node_index[(b, y)] for b, y in zip(links["b"], links["after"])],
            value=links["funds"],
            color=links["transition"].map(TRANSITION_COLOURS),
            customdata=links["transition"],
            hovertemplate="%{value} funds, %{customdata}<extra></extra>",
        ),
    )
)
fig.update_layout(title=f"Hellinger clusters ≥ {MIN_SIZE} funds, {WEEKS[0]} → {WEEKS[-1]}",
                  height=900)
fig.show()

## Splits, merges and the rest

Every cluster that did not simply survive or continue, week by week, with where its funds went (or came from) and the shares.

In [ ]:
# One row per event. Partner shares are of this cluster's funds.
def event_rows(a: str, b: str) -> list[dict]:
    rows = []
    for t in transitions[(a, b)].itertuples():
        if t.transition in ("survived", "continued"):
            continue
        week, other = (a, b) if t.side == "before" else (b, a)
        rows.append({
            "week": b,
            "transition": t.transition,
            "cluster": label(week, t.cluster),
            "partners": "; ".join(
                f"{label(other, p)} {s:.0%}" for p, s in zip(t.partners, t.shares)
            ),
        })
    return rows


events = pd.DataFrame([row for a, b in pairs for row in event_rows(a, b)])
events.set_index(["week", "transition"])

## Follow a fund

Which cluster a fund sat in each week, and what happened to that cluster next. Try any ISIN; the default sits in the mixed cluster where several environment funds (`LU0354788688`, `SE0017767577`, `LU2553958583`) and a water fund (`LU1890834598`) split off between W28 and W45.

In [ ]:
# Cluster per week for one fund.
FOLLOW_ISIN = "LU0354788688"


def next_transition(week: str, cluster: int) -> str | None:
    i = WEEKS.index(week)
    if i == len(WEEKS) - 1 or cluster == NOT_IN_CLUSTER:
        return None
    t = transitions[(week, WEEKS[i + 1])].query("side == 'before' and cluster == @cluster")
    return t["transition"].iloc[0]


pd.DataFrame([
    {
        "week": w,
        "cluster": label(w, c := int(weeks[w].points.loc[FOLLOW_ISIN, "cluster"])),
        "next week": next_transition(w, c),
    }
    for w in WEEKS if FOLLOW_ISIN in weeks[w].points.index
]).set_index("week")

## Summary: 2026-W28 → 2026-W45, Hellinger

- **Funds are steady, cluster counts are not.** 768–789 funds every week, but 35 to 49 clusters.
- **Most clusters survive a week**: 23–38 of them, depending on the week. Real splits are rare, 0–4 per week.
- **The common event is `absorbed`, then `merged` back.** In W39, 21 small clusters were folded into three big ones (`Sverige` 64 funds, a mixed environment/new-energy cluster of 48, a mixed US cluster of 58). W43 had 18 absorptions and W36/W37 had 13 each. Most of those groups come back a week or two later.
- **The environment funds flip-flop rather than split.** `LU0354788688`, `SE0017767577`, `LU2553958583` and the water fund `LU1890834598` sit in a 46–48-fund mixed cluster in W28–W35, W37, W39 and W43. In W36, W38, W40–W42 and W44–W45 they are in a small environment cluster of 5–9 funds instead. The W28 → W45 "split" in `cluster_migration.ipynb` is one side of that flip, not a one-off change in holdings.
- **Takeaway:** week-to-week labels are unstable for groups on a density border. Neighbours are the stable signal for search. Keeping map labels stable would need reference clusters with hysteresis.